# 02 Preprocessing

Week 3 objective: turn the CRMLS sold data into a cleaned dataset and prepare the time-based split plan for California single-family home close price prediction.

This notebook continues from `01_exploration.ipynb` and implements:

- missing value decisions: drop, convert invalid values to missing, or flag
- basic categorical and boolean cleaning
- row-level data quality fixes
- time-based train/test split setup using the most recent month as test
- an `X` training-window preparation table for the next baseline modeling notebook
- preprocessing pipeline template for imputation, encoding, and scaling
- cleaned CSV export

## 0. Setup


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

CLEANED_CSV = PROCESSED_DATA_DIR / "crmls_week3_cleaned.csv"
SPLIT_PLAN_CSV = PROCESSED_DATA_DIR / "crmls_week3_split_plan.csv"

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data dir: {RAW_DATA_DIR}")
print(f"Cleaned CSV: {CLEANED_CSV}")
print(f"Split plan CSV: {SPLIT_PLAN_CSV}")

Project root: /Users/soyeonpark/Projects/IDX_internship_2026/idx-california-price-prediction
Raw data dir: /Users/soyeonpark/Projects/IDX_internship_2026/idx-california-price-prediction/data/raw
Cleaned CSV: /Users/soyeonpark/Projects/IDX_internship_2026/idx-california-price-prediction/data/processed/crmls_week3_cleaned.csv
Split plan CSV: /Users/soyeonpark/Projects/IDX_internship_2026/idx-california-price-prediction/data/processed/crmls_week3_split_plan.csv


## 1. Load Raw Data

Each file is loaded with a `source_file` column so records remain traceable after combining monthly data.


In [2]:
def load_crmls_file(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)
    df["source_file"] = path.name
    return df

csv_files = sorted(RAW_DATA_DIR.glob("CRMLSSold*.csv"))

raw_df = pd.concat([load_crmls_file(path) for path in csv_files], ignore_index=True)

print(f"Loaded files: {len(csv_files)}")
print([path.name for path in csv_files])
print(f"Raw shape: {raw_df.shape}")

Loaded files: 14
['CRMLSSold202504.csv', 'CRMLSSold202505.csv', 'CRMLSSold202506.csv', 'CRMLSSold202507.csv', 'CRMLSSold202508.csv', 'CRMLSSold202509.csv', 'CRMLSSold202510.csv', 'CRMLSSold202511.csv', 'CRMLSSold202512.csv', 'CRMLSSold202601.csv', 'CRMLSSold202602.csv', 'CRMLSSold202603.csv', 'CRMLSSold202604.csv', 'CRMLSSold202605.csv']
Raw shape: (305085, 79)


## 2. Filter Project Scope and Target

- The project scope:  `Residential` + `SingleFamilyResidence`.
- The target is `ClosePrice`, so rows without a valid positive close price are dropped.


In [3]:
df = raw_df.copy()

# date parsing
df["ClosePrice"] = pd.to_numeric(df["ClosePrice"], errors="coerce")
df["CloseDate"] = pd.to_datetime(df["CloseDate"], errors="coerce")
df["close_month"] = df["CloseDate"].dt.to_period("M")
df["ListingContractDate"] = pd.to_datetime(df["ListingContractDate"], errors="coerce")

# target/date validity
df = df[df["CloseDate"].notna()]
df = df[df["ClosePrice"].notna()]
df = df[df["ClosePrice"] > 0]

# task-specific property filter
df = df[
    (df["PropertyType"] == "Residential") &
    (df["PropertySubType"].astype(str).str.replace(" ", "", regex=False) == "SingleFamilyResidence")
].copy()

df.shape

(153876, 80)

## 3. Deduplicate Listings

By EDA, we found duplicated `ListingKey` values.

In [4]:
rows_before = len(df)

df = df.sort_values(["ListingKey", "CloseDate", "source_file"])
df = df.drop_duplicates(subset=["ListingKey"], keep="last")

print(f"Duplicate ListingKey rows removed: {rows_before - len(df):,}")
print(df.shape)


Duplicate ListingKey rows removed: 112
(153764, 80)


## 4. Define Feature Decisions

In [5]:
target = "ClosePrice"
id_col = "ListingKey"
metadata_cols = [id_col, "CloseDate", "close_month", "source_file"]

continuous_numeric_cols = [
    "LivingArea",
    "LotSizeSquareFeet",
    "YearBuilt",
    "Latitude",
    "Longitude",
    "AssociationFee"
]

# Count / ordinal numeric predictors.
# These are numeric, but they are discrete counts rather than continuous measurements.
count_numeric_cols = [
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "GarageSpaces",
    "ParkingTotal",
    "Stories"
]

# Combined numeric list used for numeric summaries and correlation checks.
numeric_cols = continuous_numeric_cols + count_numeric_cols


boolean_cols = [
    "ViewYN",
    "PoolPrivateYN",
    "AttachedGarageYN",
    "FireplaceYN",
    "NewConstructionYN",
]

categorical_cols = [
    "City",
    "CountyOrParish",
    "PostalCode",
    "MLSAreaMajor",
    "Levels",
    "Flooring",
    "HighSchoolDistrict",
]

# Keep only columns that exist in the current raw files.
metadata_cols = [col for col in metadata_cols if col in df.columns]
numeric_cols = [col for col in numeric_cols if col in df.columns]
boolean_cols = [col for col in boolean_cols if col in df.columns]
categorical_cols = [col for col in categorical_cols if col in df.columns]

selected_cols = metadata_cols + [target] + numeric_cols + categorical_cols + boolean_cols
clean_df = df[selected_cols].copy()

print(f"Clean base shape: {clean_df.shape}")

Clean base shape: (153764, 28)


## 5. Clean Numeric, Boolean, and Location Values

- convert numeric columns to numeric dtype
- remove clearly non-California records
- flag invalid coordinates and set bad coordinates to missing
- convert impossible numeric values to missing
- add missing-value flags
- convert boolean fields to 0/1
- clean categorical text and mark missing categories

Imputation, scaling, rare-category grouping, and one-hot encoding are done later using the training set only.

In [6]:
clean_df = clean_df.copy()

# 1. Numeric cleaning: convert invalid strings to NaN.
for col in numeric_cols:
    clean_df[col] = pd.to_numeric(clean_df[col], errors="coerce")

# 2. Drop records that are explicitly outside the California modeling scope.
# These are not just bad coordinates; the location labels themselves are non-CA.
if "CountyOrParish" in clean_df.columns:
    non_ca_location = clean_df["CountyOrParish"].isin(["Foreign Country", "Other State"])
    print(f"Dropping non-California location rows: {non_ca_location.sum():,}")
    clean_df = clean_df.loc[~non_ca_location].copy()

# 3. Coordinate quality flag.
ca_lat_min, ca_lat_max = 32.0, 42.5
ca_lon_min, ca_lon_max = -125.0, -113.0

if {"Latitude", "Longitude"}.issubset(clean_df.columns):
    invalid_coords = (
        ~clean_df["Latitude"].between(ca_lat_min, ca_lat_max)
        | ~clean_df["Longitude"].between(ca_lon_min, ca_lon_max)
    ).fillna(False)

    clean_df["invalid_coordinates_flag"] = invalid_coords.astype(int)
    clean_df.loc[invalid_coords, ["Latitude", "Longitude"]] = np.nan
    print(f"Invalid coordinate rows flagged: {invalid_coords.sum():,}")
else:
    clean_df["invalid_coordinates_flag"] = 0

# 4. Convert impossible numeric values to missing.
if "LivingArea" in clean_df.columns:
    clean_df.loc[clean_df["LivingArea"] <= 0, "LivingArea"] = np.nan
if "BedroomsTotal" in clean_df.columns:
    clean_df.loc[clean_df["BedroomsTotal"] < 0, "BedroomsTotal"] = np.nan
if "BathroomsTotalInteger" in clean_df.columns:
    clean_df.loc[clean_df["BathroomsTotalInteger"] < 0, "BathroomsTotalInteger"] = np.nan
if "LotSizeSquareFeet" in clean_df.columns:
    clean_df.loc[clean_df["LotSizeSquareFeet"] <= 0, "LotSizeSquareFeet"] = np.nan
if "YearBuilt" in clean_df.columns:
    current_year = pd.Timestamp.today().year
    clean_df.loc[~clean_df["YearBuilt"].between(1800, current_year + 1), "YearBuilt"] = np.nan
for col in ["GarageSpaces", "ParkingTotal", "Stories", "AssociationFee"]:
    if col in clean_df.columns:
        clean_df.loc[clean_df[col] < 0, col] = np.nan

# 5. Missing flags
missing_flag_cols = []
for col in numeric_cols + boolean_cols:
    flag_col = f"{col}_missing"
    clean_df[flag_col] = clean_df[col].isna().astype(int)
    missing_flag_cols.append(flag_col)

# 6. Boolean cleaning.
def normalize_bool(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip().lower()
    if text in {"true", "t", "yes", "y", "1"}:
        return 1.0
    if text in {"false", "f", "no", "n", "0"}:
        return 0.0
    return np.nan

for col in boolean_cols:
    clean_df[col] = clean_df[col].map(normalize_bool)

# 7. Categorical text cleaning.
for col in categorical_cols:
    clean_df[col] = (
        clean_df[col]
        .astype("string")
        .str.strip()
        .replace("", "__missing__")
        .fillna("__missing__")
    )

cleaning_summary = pd.DataFrame({
    "column": numeric_cols + boolean_cols + categorical_cols,
    "missing_rate_after_cleaning": clean_df[numeric_cols + boolean_cols + categorical_cols].isna().mean().values,
})

display(cleaning_summary.sort_values("missing_rate_after_cleaning", ascending=False))

Dropping non-California location rows: 6
Invalid coordinate rows flagged: 57


,column,missing_rate_after_cleaning
5,AssociationFee,0.291673
13,AttachedGarageYN,0.121392
10,Stories,0.104756
11,ViewYN,0.090818
12,PoolPrivateYN,0.077453
15,NewConstructionYN,0.076347
8,GarageSpaces,0.039653
1,LotSizeSquareFeet,0.017846
0,LivingArea,0.001008
14,FireplaceYN,0.000774


## 6. Export Cleaned Data

In [7]:
export_df = clean_df.copy()
export_df["log_ClosePrice"] = np.log1p(export_df[target])
export_df.to_csv(CLEANED_CSV, index=False)

print(f"Saved cleaned data: {CLEANED_CSV}")

Saved cleaned data: /Users/soyeonpark/Projects/IDX_internship_2026/idx-california-price-prediction/data/processed/crmls_week3_cleaned.csv


## 7. Time-Based Split Setup

The AVM best-practices guidance recommends chronological splits and rolling-origin backtests. For the Week 4 baseline, use the latest 12 complete months as the modeling scope:

- `M12`: final test month, used once at the end
- `M10`, `M11`: historical evaluation months for choosing the training-window length
- candidate training windows: `X = 3, 6, 9` months

All candidate X values are compared on the same historical evaluation months so the comparison is fair.


In [8]:
available_months = sorted(clean_df["close_month"].dropna().unique())

if len(available_months) < 12:
    raise ValueError("Need at least 12 months for the rolling-origin split design.")

# Use the latest 12 months as the formal baseline modeling scope.
scope_months = available_months[-12:]

# M12 is the final test month. M10 and M11 are historical evaluation months.
test_month = scope_months[-1]
historical_eval_months = scope_months[-3:-1]

candidate_windows = [3, 6, 9]
max_x_for_historical_eval = min(scope_months.index(month) for month in historical_eval_months)
if max(candidate_windows) > max_x_for_historical_eval:
    raise ValueError(
        f"Not enough monthly history inside the 12-month scope for {candidate_windows}. "
        f"Maximum possible X before the first evaluation month is {max_x_for_historical_eval}."
    )

print(f"Available months: {available_months[0]} to {available_months[-1]}")
print(f"12-month modeling scope: {scope_months[0]} to {scope_months[-1]}")
print(f"Historical evaluation months: {', '.join(str(month) for month in historical_eval_months)}")
print(f"Final test month: {test_month}")
print(f"Candidate X values: {candidate_windows}")


def get_month_window(end_month, x_months):
    """Return the X months immediately before end_month within the 12-month scope."""
    end_idx = scope_months.index(end_month)
    start_idx = end_idx - x_months
    if start_idx < 0:
        raise ValueError(f"X={x_months} is not available before {end_month}.")
    return scope_months[start_idx:end_idx]


def make_rolling_split(data, x_months, eval_month):
    """Historical rolling-origin split: train on X months before eval_month, evaluate eval_month."""
    train_months = get_month_window(eval_month, x_months)
    train_df = data[data["close_month"].isin(train_months)].copy()
    eval_df = data[data["close_month"] == eval_month].copy()
    return train_df, eval_df, train_months, eval_month


def make_final_test_split(data, x_months):
    """Final split after X is chosen: train on X months before test_month, evaluate once on test_month."""
    train_months = get_month_window(test_month, x_months)
    train_df = data[data["close_month"].isin(train_months)].copy()
    test_df = data[data["close_month"] == test_month].copy()
    return train_df, test_df, train_months, test_month


Available months: 2025-04 to 2026-05
12-month modeling scope: 2025-06 to 2026-05
Historical evaluation months: 2026-03, 2026-04
Final test month: 2026-05
Candidate X values: [3, 6, 9]


## 8. Rolling-Origin Split Plan

This table prepares the Week 4 experiment design. Each candidate `X` is evaluated on the same historical months, then the selected `X` is used for the final untouched test month.


In [9]:
split_plan_rows = []

for x in candidate_windows:
    for eval_month in historical_eval_months:
        rolling_train_df, rolling_eval_df, train_months, eval_month_value = make_rolling_split(
            clean_df,
            x_months=x,
            eval_month=eval_month,
        )
        split_plan_rows.append({
            "split_type": "historical_eval",
            "X_train_months": x,
            "train_month_start": str(train_months[0]),
            "train_month_end": str(train_months[-1]),
            "train_months": ", ".join(str(month) for month in train_months),
            "eval_month": str(eval_month_value),
            "train_rows": len(rolling_train_df),
            "eval_rows": len(rolling_eval_df),
        })

    final_train_df, test_df, final_train_months, test_month_value = make_final_test_split(
        clean_df,
        x_months=x,
    )
    split_plan_rows.append({
        "split_type": "final_test_candidate",
        "X_train_months": x,
        "train_month_start": str(final_train_months[0]),
        "train_month_end": str(final_train_months[-1]),
        "train_months": ", ".join(str(month) for month in final_train_months),
        "eval_month": str(test_month_value),
        "train_rows": len(final_train_df),
        "eval_rows": len(test_df),
    })

split_plan = pd.DataFrame(split_plan_rows)
split_plan.to_csv(SPLIT_PLAN_CSV, index=False)

print(f"Saved split plan: {SPLIT_PLAN_CSV}")
display(split_plan)


Saved split plan: /Users/soyeonpark/Projects/IDX_internship_2026/idx-california-price-prediction/data/processed/crmls_week3_split_plan.csv


,split_type,X_train_months,train_month_start,train_month_end,train_months,eval_month,train_rows,eval_rows
0,historical_eval,3,2025-12,2026-02,"2025-12, 2026-01, 2026-02",2026-03,26473,11169
1,historical_eval,3,2026-01,2026-03,"2026-01, 2026-02, 2026-03",2026-04,27196,12025
2,final_test_candidate,3,2026-02,2026-04,"2026-02, 2026-03, 2026-04",2026-05,31736,12024
3,historical_eval,6,2025-09,2026-02,"2025-09, 2025-10, 2025-11, 2025-12, 2026-01, 2...",2026-03,59662,11169
4,historical_eval,6,2025-10,2026-03,"2025-10, 2025-11, 2025-12, 2026-01, 2026-02, 2...",2026-04,59382,12025
5,final_test_candidate,6,2025-11,2026-04,"2025-11, 2025-12, 2026-01, 2026-02, 2026-03, 2...",2026-05,59387,12024
6,historical_eval,9,2025-06,2026-02,"2025-06, 2025-07, 2025-08, 2025-09, 2025-10, 2...",2026-03,94906,11169
7,historical_eval,9,2025-07,2026-03,"2025-07, 2025-08, 2025-09, 2025-10, 2025-11, 2...",2026-04,94383,12025
8,final_test_candidate,9,2025-08,2026-04,"2025-08, 2025-09, 2025-10, 2025-11, 2025-12, 2...",2026-05,94298,12024


## 9. Save Cleaned Split Files

Save one file per rolling-origin train/evaluation split. Outlier thresholds are intentionally not applied here because they must be learned from each fold's training data inside the modeling notebook.


In [10]:
SPLIT_DIR = PROCESSED_DATA_DIR / "splits"
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

# Clear prior generated split CSVs so stale split files from older experiments do not confuse the Week 4 run.
for old_split_path in SPLIT_DIR.glob("*.csv"):
    old_split_path.unlink()

# Common historical evaluation and final test files.
for eval_month in historical_eval_months:
    eval_df = clean_df[clean_df["close_month"] == eval_month].copy()
    eval_path = SPLIT_DIR / f"rolling_eval_{eval_month}_cleaned.csv"
    eval_df.to_csv(eval_path, index=False)
    print(f"Saved historical evaluation set: {eval_path} ({len(eval_df):,} rows)")

test_df_common = clean_df[clean_df["close_month"] == test_month].copy()
test_path = SPLIT_DIR / "test_latest_month_cleaned.csv"
test_df_common.to_csv(test_path, index=False)
print(f"Saved final test set: {test_path} ({len(test_df_common):,} rows)")

# Candidate rolling-origin train files and final train files.
for x in candidate_windows:
    for eval_month in historical_eval_months:
        rolling_train_df, _, _, _ = make_rolling_split(clean_df, x_months=x, eval_month=eval_month)
        train_path = SPLIT_DIR / f"rolling_train_X{x}_eval_{eval_month}_cleaned.csv"
        rolling_train_df.to_csv(train_path, index=False)

    final_train_df, _, _, _ = make_final_test_split(clean_df, x_months=x)
    final_train_path = SPLIT_DIR / f"final_train_X{x}_cleaned.csv"
    final_train_df.to_csv(final_train_path, index=False)

print(f"Saved rolling-origin split files to: {SPLIT_DIR}")


Saved historical evaluation set: /Users/soyeonpark/Projects/IDX_internship_2026/idx-california-price-prediction/data/processed/splits/rolling_eval_2026-03_cleaned.csv (11,169 rows)
Saved historical evaluation set: /Users/soyeonpark/Projects/IDX_internship_2026/idx-california-price-prediction/data/processed/splits/rolling_eval_2026-04_cleaned.csv (12,025 rows)


Saved final test set: /Users/soyeonpark/Projects/IDX_internship_2026/idx-california-price-prediction/data/processed/splits/test_latest_month_cleaned.csv (12,024 rows)


Saved rolling-origin split files to: /Users/soyeonpark/Projects/IDX_internship_2026/idx-california-price-prediction/data/processed/splits


## 10. Preprocessing Pipeline Template

This section defines the preprocessing pipeline needed for modeling, but does not fit it yet. It should be fit separately inside each candidate `X` training-window experiment in the Linear Regression baseline notebook.

The pipeline handles the Week 3 preprocessing requirements:

- numeric missing values: impute with the training median
- numeric normalization: standardize numeric features with training mean/std
- categorical missing values: impute as `__missing__`
- categorical encoding: one-hot encode categories and ignore unseen test categories
- boolean and missing-flag features: pass through as already-cleaned 0/1 columns

Fitting is intentionally deferred because each `X` value has a different training set, so medians, scaler statistics, and encoder categories must be learned separately for each window.

In [11]:
def make_onehot_encoder():
    """
    Compatible with both newer and older sklearn versions.
    """
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)
    
def build_preprocessing_pipeline(
    numeric_features,
    categorical_features,
    boolean_features,
    flag_features,
    scale_numeric=True,
):
    """
    Build preprocessing pipeline.
    
    This function only defines the preprocessing steps.
    It should be fit on the training set only in the modeling notebook.
    """
    
    numeric_steps = [
        ("imputer", SimpleImputer(strategy="median")),
    ]
    
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))
    
    numeric_pipeline = Pipeline(steps=numeric_steps)
    
    categorical_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="__missing__")),
        ("onehot", make_onehot_encoder()),
    ])
    
    boolean_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
    ])
    
    flag_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
    ])
    
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_pipeline, numeric_features),
            ("cat", categorical_pipeline, categorical_features),
            ("bool", boolean_pipeline, boolean_features),
            ("flag", flag_pipeline, flag_features),
        ],
        remainder="drop",
        verbose_feature_names_out=True,
    )
    
    return preprocessor

flag_cols = [
    col for col in clean_df.columns
    if col.endswith("_missing") or col == "invalid_coordinates_flag"
]

model_feature_cols = numeric_cols + categorical_cols + boolean_cols + flag_cols

preprocessor_template = build_preprocessing_pipeline(
    numeric_features=numeric_cols,
    categorical_features=categorical_cols,
    boolean_features=boolean_cols,
    flag_features=flag_cols,
    scale_numeric=True,
)

print("Pipeline template created but not fit.")
print(f"Numeric features: {len(numeric_cols)}")
print(f"Categorical features: {len(categorical_cols)}")
print(f"Boolean features: {len(boolean_cols)}")
print(f"Flag features: {len(flag_cols)}")
print(f"Total raw model feature columns before encoding: {len(model_feature_cols)}")

Pipeline template created but not fit.
Numeric features: 11
Categorical features: 7
Boolean features: 5
Flag features: 17
Total raw model feature columns before encoding: 40


## 11. Preprocessing Summary

In [12]:
summary = {
    "raw_rows": len(raw_df),
    "cleaned_rows": len(clean_df),
    "available_month_start": str(available_months[0]),
    "available_month_end": str(available_months[-1]),
    "modeling_scope_month_start": str(scope_months[0]),
    "modeling_scope_month_end": str(scope_months[-1]),
    "historical_eval_months_for_X_selection": ", ".join(str(month) for month in historical_eval_months),
    "final_test_month": str(test_month),
    "candidate_X_values": ", ".join(map(str, candidate_windows)),
    "cleaned_columns": clean_df.shape[1],
    "cleaned_csv": str(CLEANED_CSV.relative_to(PROJECT_ROOT)),
    "split_plan_csv": str(SPLIT_PLAN_CSV.relative_to(PROJECT_ROOT)),
    "split_dir": str(SPLIT_DIR.relative_to(PROJECT_ROOT)),
}

summary_df = pd.DataFrame([summary]).T.rename(columns={0: "value"})
display(summary_df)

preprocessing_policy = pd.DataFrame([
    {
        "field_group": "Target",
        "decision": "Dropped rows where ClosePrice is missing or <= 0. Extreme high values are kept in cleaned data; fold-specific outlier thresholds are learned in modeling."
    },
    {
        "field_group": "Project scope",
        "decision": "Kept Residential / SingleFamilyResidence records only."
    },
    {
        "field_group": "Duplicate ListingKey",
        "decision": "Kept one row per ListingKey after sorting by CloseDate/source_file."
    },
    {
        "field_group": "Non-CA rows",
        "decision": "Dropped rows labeled Foreign Country or Other State."
    },
    {
        "field_group": "Coordinates",
        "decision": "Flagged invalid California coordinates and set Latitude/Longitude to missing."
    },
    {
        "field_group": "Numeric predictors",
        "decision": "Converted impossible values to missing and added missing flags; imputation is deferred."
    },
    {
        "field_group": "Boolean predictors",
        "decision": "Converted values to 0/1; remaining missing values will be imputed inside the train-only pipeline."
    },
    {
        "field_group": "Categorical predictors",
        "decision": "Cleaned text values and marked missing/empty strings as __missing__; one-hot encoding is deferred."
    },
    {
        "field_group": "Rolling-origin window selection",
        "decision": "Use the latest 12 months, compare X=3, 6, 9 on the same historical evaluation months, and reserve the latest month as final test."
    },
    {
        "field_group": "Final test split",
        "decision": "Reserve the latest month as final test and use it only once after X is selected."
    },
    {
        "field_group": "Leakage control",
        "decision": "Did not fit imputation, scaling, encoding, or outlier thresholds on the full dataset. These are fit on train only in the modeling notebook."
    },
    {
        "field_group": "Pipeline template",
        "decision": "Defined preprocessing steps for Week 4 but did not fit the pipeline in this notebook."
    },
])

display(preprocessing_policy)


,value
raw_rows,305085
cleaned_rows,153758
available_month_start,2025-04
available_month_end,2026-05
modeling_scope_month_start,2025-06
modeling_scope_month_end,2026-05
historical_eval_months_for_X_selection,"2026-03, 2026-04"
final_test_month,2026-05
candidate_X_values,"3, 6, 9"
cleaned_columns,45


,field_group,decision
0,Target,Dropped rows where ClosePrice is missing or <=...
1,Project scope,Kept Residential / SingleFamilyResidence recor...
2,Duplicate ListingKey,Kept one row per ListingKey after sorting by C...
3,Non-CA rows,Dropped rows labeled Foreign Country or Other ...
4,Coordinates,Flagged invalid California coordinates and set...
5,Numeric predictors,Converted impossible values to missing and add...
6,Boolean predictors,Converted values to 0/1; remaining missing val...
7,Categorical predictors,Cleaned text values and marked missing/empty s...
8,Rolling-origin window selection,"Use the latest 12 months, compare X=3, 6, 9 on..."
9,Final test split,Reserve the latest month as final test and use...


## Final Notes

This notebook produces the Week 3 preprocessing deliverables used by Week 4 modeling:
- cleaned base CSV
- rolling-origin split plan for candidate training windows
- historical evaluation split files
- final test split file
- candidate final train files
- preprocessing pipeline template

The cleaned CSV remains intentionally pre-imputation and pre-encoding. The modeling notebook must fit preprocessing, outlier thresholds, and the model separately inside each rolling-origin cutoff.
